## DICOM TO NPY with interpolation WBI

핵심 분석: "MATLAB과 동일한 순서"란 무엇인가?
1.MATLAB (Step E): dose = dicomread(...)를 통해 **Dose DICOM의 원본 Grid(좌표계)**를 기준으로 DVH를 계산합니다. 별도의 256x256x256 리샘플링을 거치지 않습니다.
2.현재 Python: ct_resample_crop_center 함수가 영상의 **중심(Center)**을 기준으로 새로운 2mm 격자를 생성하고, 여기에 Dose와 CT를 끼워 맞춥니다. -> 실험 목표: Python 코드에서도 "Dose 영상의 원점(Origin)과 방향(Direction)"을 기준으로 격자를 생성하도록 변경하면, MATLAB과 거의 동일한 DVH 값을 얻을 수 있습니다. (Center Crop을 끄고 Dose Grid에 정렬)

In [ ]:
#import package # CT를 기준으로 보간을 진행하는 방법 수행 
# 이런 경우 결국 CT의 첫번째 position을 가지고 만드니까 빈공간이 있긴 해도 결국 margin기준으로 만드니까 position을 그대로 사용할 수 있겠지?  - 25.01.26
import numpy as np
import os, traceback
from PIL import Image
import pydicom
from IPython.display import display
from matplotlib import pyplot as plt
import scipy.ndimage
import math
from scipy.interpolate import interpn
from scipy.interpolate import RegularGridInterpolator
import subprocess
from medpy.io import load
import glob
import csv
# for change rescaleintercept
# from pydicom import Dataset, examples
from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut
from typing import Dict, List, Tuple, Optional

import SimpleITK as sitk
from skimage import draw
import pandas as pd
from tqdm import tqdm

#image marge test
import imageio
from skimage import data
from skimage.color import rgb2gray
from PIL import Image
import random

from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.ndimage import map_coordinates
from scipy.interpolate import interp1d

In [ ]:
# set your data path
# working_dir = "FF WBI Rt"
working_dir = os.environ.get("WBI_DICOM_INPUT_ROOT", "./data/dicom")
#####
# data_root
#     ㄴ 20xx-xx__studies
#                ㄴ name _ (CT / RTST /
#  RTDOSE) folder
#                               ㄴ *.dcm                
#####
csv_output_path = "CSV/dataset_summary_2025_01_27.csv" 
output_dir  = "output_3D_NEW_matlabliek/"
npy_path = "NPY"
normalization_path = "NPY_Normalization" 

In [ ]:
# 결과 저장용 딕셔너리 초기화
all_results = {}

filtered_folder_list_CT = glob.glob(os.path.join(working_dir, '*/*_CT_*'))
filtered_folder_list_RTst = glob.glob(os.path.join(working_dir, '*/*_RTst_*'))
# filtered_folder_list_RTst = glob.glob(os.path.join("external", '*/*_RTst_*') )
# filtered_folder_list_RTDOSE = glob.glob(os.path.join(working_dir, '*/*_RTDOSE_*Accumulated*')) # for data_2025-01-21
filtered_folder_list_RTDOSE = glob.glob(os.path.join(working_dir, '*/*_RTDOSE_*'))

print(len(filtered_folder_list_CT))
print(filtered_folder_list_CT[0])
print(len(filtered_folder_list_RTst))
print(filtered_folder_list_RTst[0])
print(len(filtered_folder_list_RTDOSE))
print(filtered_folder_list_RTDOSE[0])

## start interpolation

In [ ]:
# plot_images 함수는 그대로 사용하되 extent 처리 방식 수정
def plot_images(debug, images, x_coords, y_coords, sample_image, x_sample_coords, y_sample_coords):
    if debug:
        # Create a new figure
        fig = plt.figure(figsize=(8, 4))
        # Create subplots using GridSpec for custom layout
        gs = fig.add_gridspec(1, 2, wspace=0.2)
        
        # First subplot for original CT image
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(images, cmap='gray', origin='lower', aspect='equal',
                   extent=(x_coords.min(), x_coords.max(), y_coords.min(), y_coords.max()))
        ax1.set_xlabel('R-L distance (mm)', fontsize=12)
        ax1.set_ylabel('A-P distance (mm)', fontsize=12)
        ax1.set_title('Axial Original Image : ' + str(images.shape), fontsize=12)
        ax1.set_aspect('equal', 'box')
        ax1.invert_yaxis()
    
        # Second subplot for reshaped CT sample image
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.imshow(sample_image, cmap='gray', origin='lower', aspect='equal',
                   extent=(x_sample_coords.min(), x_sample_coords.max(), y_sample_coords.min(), y_sample_coords.max()))
        ax2.set_xlabel('R-L distance (mm)', fontsize=12)
        ax2.set_ylabel('A-P distance (mm)', fontsize=12)
        ax2.set_title('Axial Sample Image : ' + str(sample_image.shape), fontsize=12)
        ax2.set_aspect('equal', 'box')
        ax2.invert_yaxis()
        
        plt.show()

In [ ]:
def update_csv_row(csv_path, ct_name, unit, contour_list_str, process_val="O"):
    """
    csv_path 파일을 모두 읽어,
    (ct_name, unit, contour_list_str) 가 일치하는 행을 찾아
    그 행의 4번째 컬럼(process)에 process_val("O"/"X")을 쓰고,
    전체를 다시 csv_path로 덮어쓴다.
    """
    rows = []
    updated = False  # 수정 여부 확인

    # 1) 기존 CSV 전체 읽어오기
    with open(csv_path, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        # 맨 첫 줄은 헤더
        header = next(reader)
        rows.append(header)  # header 저장

        # 나머지 본문
        for row in reader:
            if row[0] == ct_name and row[1] == unit:
                # (ct_name, unit) 일치하는 행 찾기
                row[-1] = process_val  # process 컬럼 업데이트
                updated = True
            rows.append(row)

    if not updated:
        print(f"No matching row found for ct_name={ct_name}, unit={unit}.")

    # 2) 수정된 rows를 다시 CSV로 쓰기 (덮어쓰기)
    with open(csv_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerows(rows)


In [ ]:
def get_ctv_wb_mask_high_res(rtstruct_path, ct_dcm_list, roi_name):
    """
    CT DICOM 리스트(3D)를 기준으로 RTSTRUCT의 특정 ROI를 고해상도 3D 마스크로 변환합니다.
    """
    # 1. RTSTRUCT 로드
    ds_rt = pydicom.dcmread(rtstruct_path)
    
    # 2. ROI Number 찾기
    roi_number = None
    for roi in ds_rt.StructureSetROISequence:
        if roi.ROIName == roi_name:
            roi_number = roi.ROINumber
            break
            
    if roi_number is None:
        # print(f"ROI Name '{roi_name}' not found in RTSTRUCT.")
        # ROI가 없으면 빈 3D 마스크 반환
        return np.zeros((len(ct_dcm_list), ct_dcm_list[0].Rows, ct_dcm_list[0].Columns), dtype=np.float32)

    # 3. 빈 3D 마스크 생성 (Z, Y, X)
    nz = len(ct_dcm_list)
    ny = ct_dcm_list[0].Rows
    nx = ct_dcm_list[0].Columns
    
    # Soft Masking을 위해 float32 사용 (필요시 uint8로 변경 가능)
    mask_3d = np.zeros((nz, ny, nx), dtype=np.float32)

    # 4. CT 슬라이스의 Z 좌표와 인덱스 매핑 (오차 허용을 위해 round 사용)
    # CT 리스트는 이미 Z축 정렬되어 있다고 가정
    z_to_slice_idx = {}
    for idx, ct_ds in enumerate(ct_dcm_list):
        z_pos = float(ct_ds.ImagePositionPatient[2])
        z_to_slice_idx[round(z_pos, 2)] = idx  # 소수점 2자리 반올림하여 키로 사용
        
        # 픽셀 정보 미리 캐싱 (모든 슬라이스가 같다고 가정)
        if idx == 0:
            origin = np.array(ct_ds.ImagePositionPatient)
            spacing = np.array(ct_ds.PixelSpacing) # [row(y), col(x)]

    # 5. ROI Contour 좌표를 마스크에 그리기
    found_contours = False
    for contour in ds_rt.ROIContourSequence:
        if contour.ReferencedROINumber == roi_number:
            if not hasattr(contour, 'ContourSequence'):
                break # 컨투어 데이터가 없는 경우
                
            found_contours = True
            for point_seq in contour.ContourSequence:
                # 컨투어 포인트 추출 (x, y, z)
                points = np.array(point_seq.ContourData).reshape(-1, 3)
                
                # 현재 컨투어의 Z 좌표 (첫 번째 포인트 기준)
                z_current = points[0, 2]
                
                # 해당하는 CT 슬라이스 인덱스 찾기
                slice_idx = z_to_slice_idx.get(round(z_current, 2))
                
                if slice_idx is not None:
                    # 물리적 좌표(mm) -> 픽셀 좌표(index) 변환
                    # pixel_x = (world_x - origin_x) / spacing_x
                    r_idx = (points[:, 1] - origin[1]) / spacing[0] # Row (Y)
                    c_idx = (points[:, 0] - origin[0]) / spacing[1] # Col (X)
                    
                    # 다각형 내부 채우기 (skimage.draw.polygon 사용)
                    rr, cc = draw.polygon(r_idx, c_idx, shape=(ny, nx))
                    
                    # 마스크에 1.0 할당 (Binary Mask)
                    mask_3d[slice_idx, rr, cc] = 1.0

    return mask_3d

In [ ]:
def get_dose_interpolator(ds_rtdose):
    """
    RTDOSE 객체로부터 물리적 좌표계를 생성하고 보간기(Interpolator)를 반환합니다.
    (SliceThickness가 없거나 None인 경우에 대한 방어 코드 추가)
    """
    # 1. Dose Grid Scaling & Unit conversion
    dose_array = ds_rtdose.pixel_array.astype(np.float32) * float(ds_rtdose.DoseGridScaling)
    
    units = getattr(ds_rtdose, "DoseUnits", "").upper()
    if units == "CGY":
        dose_array /= 100.0  # Convert to Gy
    
    # 2. Construct Physical Coordinates of Dose Grid
    ipp = np.array(ds_rtdose.ImagePositionPatient, dtype=np.float32)
    ps = np.array(ds_rtdose.PixelSpacing, dtype=np.float32) # [row_spacing, col_spacing] -> [y, x]
    
    # [수정된 부분] SliceThickness가 None이거나 없을 때 처리
    st = None
    if hasattr(ds_rtdose, 'SliceThickness') and ds_rtdose.SliceThickness is not None:
        try:
            st = float(ds_rtdose.SliceThickness)
        except ValueError:
            st = None # 값이 비어있는 문자열 '' 인 경우 등 처리

    # SliceThickness를 못 구했다면 GridFrameOffsetVector에서 계산
    if (st is None or st == 0) and hasattr(ds_rtdose, 'GridFrameOffsetVector'):
        frames = ds_rtdose.GridFrameOffsetVector
        if len(frames) > 1:
            st = float(frames[1] - frames[0])
    
    # 그래도 없다면 임시로 ps[0] 사용 (드문 경우)
    if st is None:
        st = ps[0]

    nz, ny, nx = dose_array.shape
    
    # Z, Y, X 축 좌표 생성
    if hasattr(ds_rtdose, 'GridFrameOffsetVector'):
        z_coords = ipp[2] + np.array(ds_rtdose.GridFrameOffsetVector, dtype=np.float32)
    else:
        z_coords = ipp[2] + np.arange(nz) * st
        
    y_coords = ipp[1] + np.arange(ny) * ps[0]
    x_coords = ipp[0] + np.arange(nx) * ps[1]
    
    # 3. Create Interpolator
    interpolator = RegularGridInterpolator(
        (z_coords, y_coords, x_coords), 
        dose_array, 
        method='linear', 
        bounds_error=False, 
        fill_value=0.0
    )
    
    return interpolator

def get_dose_values_on_mask(interpolator, mask, ct_info):
    """
    고해상도 Mask 내부의 좌표들에 대해 Dose 값을 보간하여 추출합니다.
    Returns: 1D numpy array of dose values (Gy)
    """
    # Mask가 비어있으면 빈 배열 반환
    if np.sum(mask) == 0:
        return np.array([], dtype=np.float32)
    
    # 1. Mask(High-Res)가 1인 인덱스 추출
    # z_idx, y_idx, x_idx = np.where(mask > 0)
    # 메모리 효율을 위해 where로 인덱스만 가져옴
    indices = np.argwhere(mask > 0) # shape: (N, 3) -> [z, y, x]
    
    # 2. 인덱스를 물리적 좌표(Physical Coordinates)로 변환
    # CT의 Origin, Spacing 사용
    origin = ct_info['origin']   # [x, y, z]
    spacing = ct_info['spacing'] # [x, y, z] -> 주의: 보통 numpy는 [z, y, x] 순서 헷갈림 주의
    
    # CT info spacing이 [z_spacing, y_spacing, x_spacing] 이라고 가정 (일반적 DICOM numpy 순서)
    # 하지만 DICOM tag PixelSpacing은 [row(y), col(x)] 순서임. 확인 필요.
    # 여기서는 ct_info가 (z, y, x) 순서로 정리되어 있다고 가정합니다.
    
    z_phys = origin[2] + indices[:, 0] * spacing[0] # z index * z spacing
    y_phys = origin[1] + indices[:, 1] * spacing[1] # y index * y spacing
    x_phys = origin[0] + indices[:, 2] * spacing[2] # x index * x spacing
    
    # 3. Interpolator에 넣기 위해 좌표 합치기 (N, 3)
    query_points = np.stack((z_phys, y_phys, x_phys), axis=1)
    
    # 4. Interpolation 실행
    dose_values = interpolator(query_points)
    
    return dose_values.astype(np.float32)

# ==============================================================================
# 2. Metric Calculations (Updated for 1D Array)
# ==============================================================================
# 이제 입력 dose_arr는 3D 그리드가 아니라, 마스크 내부의 값들만 모은 1D 배열입니다.

def compute_cumulative_dvh_1d(dose_vals: np.ndarray, bin_width_gy: float = 0.1, max_dose_gy: float = None):
    if dose_vals.size == 0:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32)

    if max_dose_gy is None:
        max_dose_gy = float(dose_vals.max())

    bins = np.arange(0.0, max_dose_gy + bin_width_gy, bin_width_gy, dtype=np.float32)
    s = np.sort(dose_vals)
    idx = np.searchsorted(s, bins, side="left")
    vol_pct = (s.size - idx) / s.size * 100.0
    return bins, vol_pct.astype(np.float32)

def v_at_1d(dose_vals: np.ndarray, thr_gy: float) -> float:
    if dose_vals.size == 0:
        return float("nan")
    return float((dose_vals >= thr_gy).mean() * 100.0)

def dmean_1d(dose_vals: np.ndarray) -> float:
    if dose_vals.size == 0:
        return float("nan")
    return float(dose_vals.mean())

def dmax_1d(dose_vals: np.ndarray) -> float:
    if dose_vals.size == 0:
        return float("nan")
    return float(dose_vals.max())

def infer_wbi_side(case_id: str):
    u = (case_id or "").upper()
    if "WL" in u: return "L"
    if "WR" in u: return "R"
    return None

# ==============================================================================
# 3. Main Function
# ==============================================================================

def compute_and_save_origin_dicom_dvh(
    out_csv_path: str,
    study_id: str,
    case_id: str,
    rtstruct_path: str,
    ds_rtdose,
    ct_dcm_list,  # <--- [중요] CT DICOM 리스트가 추가되어야 합니다 (고해상도 기준)
    prescription_gy: float,
    roi_name_map: dict,
    bin_width_gy: float = 0.1,
    include_full_dvh: bool = True,
):
    """
    origin DICOM DVH (High-Precision / MATLAB Equivalent):
    - dose: Interpolated from ds_rtdose onto CT grid
    - mask: Generated from RTSTRUCT based on CT grid (High Res)
    """
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)

    # 1. Prepare Dose Interpolator
    dose_interpolator = get_dose_interpolator(ds_rtdose)

    # 2. Extract CT Grid Info (for Coordinate Mapping)
    # CT 리스트는 Z축으로 정렬되어 있어야 함
    ct_dcm_list.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    
    first_ct = ct_dcm_list[0]
    ct_spacing = np.array([float(first_ct.SliceThickness), float(first_ct.PixelSpacing[1]), float(first_ct.PixelSpacing[0])]) # Z, Y, X
    ct_origin = np.array([float(first_ct.ImagePositionPatient[2]), float(first_ct.ImagePositionPatient[1]), float(first_ct.ImagePositionPatient[0])]) # Z, Y, X
    
    ct_info = {'origin': ct_origin, 'spacing': ct_spacing}

    # 3. Side Inference
    side = infer_wbi_side(case_id)
    if side == "L":
        ipsi_lung_key, contra_lung_key = "Lung_L", "Lung_R"
    elif side == "R":
        ipsi_lung_key, contra_lung_key = "Lung_R", "Lung_L"
    else:
        ipsi_lung_key, contra_lung_key = "Lung_L", "Lung_R"

    # 4. Calculate Metrics per ROI
    # ROI별로 Mask를 만들고 -> Dose 값을 추출(보간) -> Metric 계산
    
    metrics = []
    dvh_data = {} # For saving full DVH later

    v95_thr = prescription_gy * 0.95
    dmax_lim = prescription_gy * 1.07

    # 처리할 ROI 리스트 정의
    target_rois = ["CTV", "Heart", ipsi_lung_key, contra_lung_key, "Contra_Breast"]
    
    for roi_key in target_rois:
        # roi_name_map에 해당 키가 없으면 스킵
        if roi_key not in roi_name_map and roi_key not in ["CTV", "Heart", "Contra_Breast"]:
             # lung key 같은 경우는 roi_name_map에 없을 수도 있으니 체크
             pass

        # 실제 DICOM ROI 이름 찾기
        dicom_roi_name = roi_name_map.get(roi_key)
        if not dicom_roi_name:
            # Lung_L, Lung_R 같은 경우 맵핑이 직접 안되어있으면 키 그대로 시도해볼 수 있음 (상황에 따라)
            if roi_key in roi_name_map.values(): # 값이면
                 dicom_roi_name = roi_key 
            else:
                 continue # 맵핑 없으면 스킵

        # [중요] High-Res Mask 생성 (CT 기준)
        # 사용자 정의 함수 get_mask_from_rtstruct가 필요함 (여기서는 rt_utils 등을 쓴다고 가정하거나 구현 필요)
        # 기존 get_ctv_wb_mask 대신 CT 기반 마스크 생성 함수 호출
        # mask_high_res = get_mask_on_ct_grid(rtstruct_path, ct_dcm_list, dicom_roi_name) 
        
        # *사용자의 get_ctv_wb_mask가 CT객체를 받을 수 있도록 수정되었다고 가정하거나,*
        # *아래와 같이 호출해야 함. 여기서는 함수명을 직관적으로 변경함*
        try:
             # 여기서 ct_dcm_list[0] 등을 넘겨서 Shape과 Affine 정보를 줌
            #  mask_high_res = get_ctv_wb_mask_high_res(rtstruct_path, ct_dcm_list, dicom_roi_name)
            mask_high_res = get_ctv_wb_mask_high_res(rtstruct_path, ct_dcm_list, dicom_roi_name)
        except Exception as e:
            print(f"Skipping {roi_key}: {e}")
            continue

        # Dose Interpolation (핵심!)
        dose_vals_1d = get_dose_values_on_mask(dose_interpolator, mask_high_res, ct_info)
        dvh_data[roi_key] = dose_vals_1d # 나중에 DVH 저장을 위해 보관

        # Calculate Specific Metrics
        if roi_key == "CTV":
            metrics += [
                ("CTV", "V95_%", v_at_1d(dose_vals_1d, v95_thr), "%", ">=95", lambda x: x >= 95.0),
                ("CTV", "Dmax_Gy", dmax_1d(dose_vals_1d), "Gy", f"<= {dmax_lim:.2f}", lambda x: x <= dmax_lim),
            ]
        elif roi_key == "Heart":
             metrics += [
                ("Heart", "V7_%", v_at_1d(dose_vals_1d, 7.0), "%", "<3", lambda x: x < 3.0),
                ("Heart", "V1.5_%", v_at_1d(dose_vals_1d, 1.5), "%", "<30", lambda x: x < 30.0),
            ]
        elif roi_key == ipsi_lung_key:
             metrics += [
                (f"Ipsi_Lung({ipsi_lung_key})", "V8_%", v_at_1d(dose_vals_1d, 8.0), "%", "<15", lambda x: x < 15.0),
            ]
        elif roi_key == contra_lung_key:
             metrics += [
                (f"Contra_Lung({contra_lung_key})", "Dmean_Gy", dmean_1d(dose_vals_1d), "Gy", "<2", lambda x: x < 2.0),
            ]
        elif roi_key == "Contra_Breast":
             metrics += [
                ("Contra_Breast", "Dmean_Gy", dmean_1d(dose_vals_1d), "Gy", "<=3", lambda x: x <= 3.0),
            ]

    # ===== CSV write =====
    fieldnames = ["kind","study_id","case_id","roi","dose_gy","vol_pct","metric","value","unit","constraint","pass"]
    rows = []

    # metric rows
    for roi, metric_name, val, unit, cons, rule in metrics:
        ok = rule(val) if np.isfinite(val) else False
        rows.append({
            "kind":"METRIC",
            "study_id":study_id,
            "case_id":case_id,
            "roi":roi,
            "dose_gy":"",
            "vol_pct":"",
            "metric":metric_name,
            "value":float(val) if np.isfinite(val) else "",
            "unit":unit,
            "constraint":cons,
            "pass":str(ok)
        })

    # dvh rows
    if include_full_dvh:
        for key, vals in dvh_data.items():
            bins, vol_pct = compute_cumulative_dvh_1d(vals, bin_width_gy=bin_width_gy)
            for d, v in zip(bins, vol_pct):
                rows.append({
                    "kind":"DVH",
                    "study_id":study_id,
                    "case_id":case_id,
                    "roi":key,
                    "dose_gy":float(d),
                    "vol_pct":float(v),
                    "metric":"",
                    "value":"",
                    "unit":"",
                    "constraint":"",
                    "pass":""
                })

    with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"[origin DVH] saved -> {out_csv_path}")

In [ ]:
X = 0
Y = 1
Z = 2
visable_debug =  False # if you want see plot chart or image just keep True 

target_x_axis = 256
target_y_axis = 256
target_z_axis = 256

new_spacing = (2.0, 2.0, 2.0)

clip_min = -300
clip_max = 800

target_value = 26.00 * 1.1 # 처방선량 110%

# 원하는 MHA_INDEX 리스트
selected_indices = [
    "CTV_WB.mha", 
    "A_LAD.mha", 
    "Contra_Breast.mha", 
    "Contra_Lung.mha", 
    "External.mha", 
    "Heart.mha", 
    "Ipsi_Lung.mha",
    "RING.mha"
]

# 원하는 순서 정의 (lad → ctv_wb → contra_breast → external → heart → ipsi_lung → contra_lung)
desired_order = [
    "A_LAD.mha",
    "CTV_WB.mha",
    "Contra_Breast.mha",
    "External.mha",
    "Heart.mha",
    "Ipsi_Lung.mha",
    "Contra_Lung.mha",
    "RING.mha"
]

def order_func(x: str) -> int:
    """
    x: 파일 이름 (예: "lad.mha", "heart.mha" 등)
    원하는 순서의 index를 반환하고,
    만약 해당 이름이 desired_order에 없는 경우 999 반환
    """
    for i, item in enumerate(desired_order):
        if item in x.lower():
            return i
    return 999

# creatFolder("CSV")
# # CSV 파일에 헤더 작성
# with open(csv_output_path, mode='w', newline='') as file:
#     writer = csv.writer(file)
#     #writer.writerow(['name', 'unit', 'contour_list', 'process'])  # 헤더 작성
#     writer.writerow(['name', 'unit(MRN)'] + desired_order + ['process'])

In [ ]:
# 기존 ct_resample_crop_center 대신 사용할, Dose 기준 정렬 함수를 셀에 추가
def ct_resample_align_to_dose(
    input_image: sitk.Image,
    dose_image: sitk.Image,  # 기준이 될 Dose 영상
    new_size=(256, 256, 256),
    new_spacing=(2.0, 2.0, 2.0)
):
    # 1. Dose의 좌표계 정보 가져오기 (MATLAB과 일치시키기 위함)
    # Dose의 원점(Origin)을 그대로 사용
    align_origin = dose_image.GetOrigin()
    align_direction = dose_image.GetDirection()
    
    # 2. Resample 설정
    resample_filter = sitk.ResampleImageFilter()
    resample_filter.SetSize(new_size)
    resample_filter.SetOutputSpacing(new_spacing)
    resample_filter.SetOutputOrigin(align_origin)      # <--- 중요: Dose Origin 사용
    resample_filter.SetOutputDirection(align_direction)# <--- 중요: Dose Direction 사용
    resample_filter.SetInterpolator(sitk.sitkLinear)
    resample_filter.SetDefaultPixelValue(-1000) # CT air value

    # 3. 실행
    output_image = resample_filter.Execute(input_image)
    
    # 좌표 계산 (Optional)
    rx = align_origin[0] + np.arange(new_size[0]) * new_spacing[0]
    ry = align_origin[1] + np.arange(new_size[1]) * new_spacing[1]
    rz = align_origin[2] + np.arange(new_size[2]) * new_spacing[2]

    return output_image, rx, ry, rz

In [ ]:
def ct_resample_crop_center(
    input_image: sitk.Image,
    new_size=(256, 256, 256),
    new_spacing=(2.0, 2.0, 2.0),
    use_center_crop=True  # <--- [NEW] True면 기존처럼 중심 정렬, False면 원점 유지
):
    print(f"ct_resample start (Center Crop: {use_center_crop})")
    
    # 1) 원본 정보
    original_size = np.array(input_image.GetSize(), dtype=np.int32)
    original_spacing = np.array(input_image.GetSpacing(), dtype=np.float64)
    original_direction = np.array(input_image.GetDirection()).reshape(3,3)
    original_origin = np.array(input_image.GetOrigin(), dtype=np.float64)
    
    # (디버그용 배열 변환 로직은 생략하거나 그대로 두셔도 됩니다)
    
    # 2) 원본 FOV 및 중심 계산
    fov_original = original_size * original_spacing
    half_fov_original_dir = original_direction @ (0.5 * fov_original)
    original_center = original_origin + half_fov_original_dir

    # 3) 최종 리샘플할 크기
    final_new_size = (new_size[0], new_size[1], new_size[2])    

    # 새 영상 물리적 크기(FOV)
    fov_new = np.array(final_new_size) * np.array(new_spacing)
    half_fov_new_dir = original_direction @ (0.5 * fov_new)

    # ====================================================
    # 4) 새 origin 계산 [수정된 부분]
    # ====================================================
    if use_center_crop:
        # 기존 로직: 새 FOV의 중심을 원본 이미지 중심에 맞춤
        new_origin = (original_center - half_fov_new_dir).tolist()
    else:
        # 변경 로직: 원본 Origin을 그대로 사용 (MATLAB/TPS와 좌표 일치율 높임)
        # 단, 원본보다 새 FOV가 작으면 잘릴 수 있고, 크면 padding 됨
        new_origin = original_origin.tolist()
        
        # (옵션) 만약 Z축 등을 원본 범위에 딱 맞추려면 new_size를 조절해야 할 수도 있음
        # 여기서는 요청하신 '2mm 고정'을 위해 Origin만 고정하고 진행
    # ====================================================

    # 5) Resample 설정
    resample_filter = sitk.ResampleImageFilter()
    resample_filter.SetSize(list(final_new_size))
    resample_filter.SetOutputSpacing(new_spacing)
    resample_filter.SetOutputOrigin(new_origin)
    resample_filter.SetOutputDirection(input_image.GetDirection())
    resample_filter.SetInterpolator(sitk.sitkLinear)
    resample_filter.SetDefaultPixelValue(0) # CT 배경값 (Hounsfield Unit에 따라 -1000 등으로 조정 필요할 수 있음)

    # 6) 실제 리샘플 실행
    output_image = resample_filter.Execute(input_image)

    # ... (이하 배열 변환 및 Plot 코드는 기존과 동일) ...
    
    # 좌표 계산 (리턴용)
    rx_coords = new_origin[0] + np.arange(final_new_size[0]) * new_spacing[0]
    ry_coords = new_origin[1] + np.arange(final_new_size[1]) * new_spacing[1]
    rz_coords = new_origin[2] + np.arange(final_new_size[2]) * new_spacing[2]

    return output_image, rx_coords, ry_coords, rz_coords

In [ ]:
def resample_dose_to_ct(dose_image, reference_image, x_sample_coords, y_sample_coords, z_sample_coords, dose_grid_scaling):
    """
    dose_image를 reference_image(예: 리샘플된 CT)의 공간 정보에 맞춰
    동일하게 리샘플링하는 함수 예시입니다.
    """
    # 1) 리샘플링 필터 선언
    resample_filter = sitk.ResampleImageFilter()
    
    # 2) Reference(기준이 되는 CT 리샘플 결과) 정보 설정
    #    CT 리샘플에 사용한 spacing, origin, direction, size를 그대로 사용합니다.
    resample_filter.SetReferenceImage(reference_image)
    
    # 3) 보간 방법 설정 (선형 보간, 최근린이웃 등)
    #    RTDOSE인 경우 일반적으로 Linear 보간을 사용하지만,
    #    필요에 따라 NearestNeighbor 등을 선택할 수 있습니다.
    resample_filter.SetInterpolator(sitk.sitkLinear)
    
    # 4) (옵션) 디폴트 값 설정
    #    영상 밖(배경) 영역에 들어갈 값입니다. 보통 0.0 등을 사용.
    resample_filter.SetDefaultPixelValue(0.0)
    
    # 5) 실제 리샘플 실행
    resampled_dose = resample_filter.Execute(dose_image)

    # 6) 원본 dose 파일 읽기
    print(f"Image Size: {dose_image.GetSize()}")
    print(f"Pixel Spacing: {dose_image.GetSpacing()}")
    print(f"Image Origin: {dose_image.GetOrigin()}")
    print(f"Image Direction: {dose_image.GetDirection()}")
    print(f"Image Pixel Type: {sitk.GetPixelIDValueAsString(dose_image.GetPixelID())}")
    origin_dose_array = sitk.GetArrayFromImage(dose_image)  # (z, y, x)
    origin_dose_array = origin_dose_array.transpose(2,1,0)
    origin_dose_array = np.rot90(origin_dose_array, 3)
    origin_dose_array = np.fliplr(origin_dose_array)

    original_spacing = dose_image.GetSpacing()  # (x, y, z)
    x_coords = np.arange(origin_dose_array.shape[0]) * original_spacing[0]
    y_coords = np.arange(origin_dose_array.shape[1]) * original_spacing[1]

    resampled_rtdose_array = sitk.GetArrayFromImage(resampled_dose) # (z x y)
    resampled_rtdose_array = resampled_rtdose_array.transpose(2,1,0)
    resampled_rtdose_array = np.rot90(resampled_rtdose_array, 3)
    resampled_rtdose_array = np.fliplr(resampled_rtdose_array)
    max_value = np.max(resampled_rtdose_array)
    print("before : ", max_value)
    # #### dose max 26.00 Gy 이상이 아니면 skip ######
    # if max_value < 26.0:
    #     print("Dose max value is less than 26.0 Gy. Skipping this case.")
    #     return np.zeros_like(resampled_rtdose_array)[..., np.newaxis]
    
    resampled_rtdose_array = resampled_rtdose_array * dose_grid_scaling
    max_value = np.max(resampled_rtdose_array)
    print("after : ", max_value)
    resampled_rtdose_array = resampled_rtdose_array/target_value
    max_value = np.max(resampled_rtdose_array)
    print("normalization_after : ", max_value)
    resampled_rtdose_array_expend = np.expand_dims(resampled_rtdose_array, axis = 3)

    # Plot 호출
    plot_images(
        visable_debug,
        origin_dose_array[:, :, 90],  # 원본 CT 이미지
        x_coords,
        y_coords,
        resampled_rtdose_array[:, :, 95],  # 리샘플된 CT 이미지
        x_sample_coords,
        y_sample_coords
    )
    
    return resampled_rtdose_array_expend

In [ ]:
slice_part_name = ['CT', 'A_LAD.mha', 'CTV.mha', 'Contra_Breast.mha', 'External.mha', 'Heart.mha', 'Lung_L.mha', 'Lung_R.mha', "RING.mha", 'RTDOSE']

# Step 3. V95% 계산
def calc_v95(dose, mask, threshold):
    """
    Weighted V95 Calculation for Soft Masks
    dose: Dose array
    mask: Soft mask array (0.0 ~ 1.0 float)
    threshold: Dose threshold (e.g., 26.0 * 0.95)
    """
    # 1. 마스크(부피) 총합 계산 (실수형 가중치 합산)
    total_volume_weight = np.sum(mask)
    
    if total_volume_weight == 0:
        return 0.0

    # 2. 임계값(threshold) 이상인 복셀들의 마스크 가중치 합산
    # dose가 threshold 이상인 위치의 mask 값들만 더합니다.
    target_volume_weight = np.sum(mask[dose >= threshold])
    
    # 3. 비율 계산
    v95 = (target_volume_weight / total_volume_weight) * 100.0
    return v95

def rasterize_polygon(xy, origin, spacing, shape):
    # 2D polygon → 2D mask
    from skimage.draw import polygon
    coords = (xy - origin) / spacing
    rr, cc = polygon(coords[:,1], coords[:,0], shape)
    mask = np.zeros(shape, dtype=bool)
    mask[rr, cc] = 1
    return mask

def get_ctv_wb_mask(rtstruct_path, ref_ds, roi_name='CTV_WB'):
    struct = pydicom.dcmread(rtstruct_path)
    structures = {roi.ROINumber: roi.ROIName for roi in struct.StructureSetROISequence}
    # ROI 번호 찾기
    ctv_wb_number = None
    for num, name in structures.items():
        print(f"ROI Number: {num}, Name: {name}")
        # if name.upper() == roi_name.upper():
        if roi_name.upper() in name.upper():  # 부분 포함 조건
            ctv_wb_number = num
            break
    if ctv_wb_number is None:
        raise ValueError(f"{roi_name} ROI not found!")
    # Contour Sequence에서 해당 ROI만 추출
    for roi_contour in struct.ROIContourSequence:
        if roi_contour.ReferencedROINumber == ctv_wb_number:
            contours = []
            for contour_seq in roi_contour.ContourSequence:
                contours.append(np.array(contour_seq.ContourData).reshape(-1, 3))
            break
    else:
        raise ValueError("CTV_WB contour not found!")
    # Dose grid 기준 마스크 생성 (SimpleITK 활용)
    dose_shape = ref_ds.pixel_array.shape
    img_pos = np.array(ref_ds.ImagePositionPatient)
    spacing = list(ref_ds.PixelSpacing) + [float(ref_ds.GridFrameOffsetVector[1] - ref_ds.GridFrameOffsetVector[0])]
    mask = np.zeros(dose_shape, dtype=bool)
    for contour in contours:
        # SimpleITK로 z축마다 2D polygon rasterize
        z_indices = np.round((contour[:,2] - img_pos[2]) / spacing[2]).astype(int)
        for z in np.unique(z_indices):
            xy = contour[z_indices == z][:,:2]
            if len(xy) < 3: continue
            slice_mask = rasterize_polygon(xy, img_pos[:2], spacing[:2], dose_shape[1:])
            if 0 <= z < dose_shape[0]:
                mask[z][slice_mask] = 1
    return mask

def load_dvh_npy(file):
    data = np.load(file)
    print("dose max : ", data[:,:,:,9].max()) 
    print(f"Loaded {file} with {data.shape} ROIs")
    
    if data.shape[-1] == len(slice_part_name):
        try:
            ctv_idx = slice_part_name.index('CTV.mha')
            dose_idx = slice_part_name.index('RTDOSE')
            
            # [수정 1] 마스크를 bool로 바꾸지 않고 float 그대로 가져옴 (Soft Mask 유지)
            # 필요시 0.0~1.0 사이 값인지 확인 (Clip 등)
            ctv_mask_soft = data[..., ctv_idx].astype(np.float32) 
            
            dose = data[..., dose_idx]
            dose = dose * (26.0 * 1.1) 
            prescription = 26.0
            v95_threshold = prescription * 0.95
            
            # [수정 2] 위에서 만든 Weighted calc_v95 함수 사용 (또는 직접 계산)
            total_vol = np.sum(ctv_mask_soft)

            if total_vol == 0:
                v95 = 0
            else:
                # Dose가 조건 만족하는 위치의 "Soft Mask 값"을 더함
                v95 = np.sum(ctv_mask_soft[dose >= v95_threshold]) / total_vol * 100
                dose_mean = np.sum(dose * ctv_mask_soft) / total_vol
            
            print(f"변환후 CTV 계산값 {os.path.basename(file)} | CTV V95%: {v95:.2f}%")
            print(f"CTV Dose Mean: {dose_mean:.2f} Gy") # 결과 출력

            # 1. 확실한 CTV 영역 정의 (보통 0.5 기준)
            binary_mask = ctv_mask_soft > 0.5

            # 2. 해당 영역 내에서 최대 선량 찾기
            if np.any(binary_mask):  # 마스크 영역이 존재하는지 확인
                d_max = np.max(dose[binary_mask])
            else:
                d_max = 0.0

            print(f"CTV Dmax: {d_max:.2f} Gy")
            
            if v95 < 95:
                print("check value! : ", v95)
                
        except Exception as e:
            print(f"Error calculating V95 for {file}: {e}")

In [ ]:
def resample_mask_to_reference(mask_img: sitk.Image, reference_img: sitk.Image):
    # mask는 무조건 nearest neighbor로 리샘플링
    return sitk.Resample(
        mask_img,
        reference_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0.0,
        sitk.sitkUInt8
    )

In [ ]:
def resample_dose_to_reference(dose_img: sitk.Image, reference_img: sitk.Image, interp="linear", default_value=0.0):
    if interp == "linear":
        itp = sitk.sitkLinear
    elif interp == "bspline":
        itp = sitk.sitkBSpline
    else:
        raise ValueError("interp must be 'linear' or 'bspline'")

    return sitk.Resample(
        dose_img,
        reference_img,
        sitk.Transform(),
        itp,
        float(default_value),
        sitk.sitkFloat32
    )

In [ ]:
prescription = 26.0
v95_threshold = prescription * 0.95  # 24.7 Gy

# =========================
# 0) RTST / RTDOSE: UID->파일경로 인덱싱 (루프 밖 1회)
# =========================
def build_uid_index_for_rtst(rtst_folders):
    uid2rtst = {}
    for rtst_folder in rtst_folders:
        dcm_files = [f for f in os.listdir(rtst_folder) if f.endswith(".dcm")]
        if not dcm_files:
            continue
        rtst_file = os.path.join(rtst_folder, dcm_files[0])
        ds = pydicom.dcmread(rtst_file, stop_before_pixels=True, force=True)
        uid = getattr(ds, "StudyInstanceUID", None)
        if uid:
            uid2rtst[uid] = rtst_file
    return uid2rtst

def build_uid_index_for_rtdose(rtdose_folders):
    uid2dose = {}
    for dose_folder in rtdose_folders:
        files = os.listdir(dose_folder)
        if not files:
            continue
        dose_file = os.path.join(dose_folder, files[0])
        ds = pydicom.dcmread(dose_file, stop_before_pixels=True, force=True)
        uid = getattr(ds, "StudyInstanceUID", None)
        if uid:
            uid2dose[uid] = dose_file
    return uid2dose

uid2rtst = build_uid_index_for_rtst(filtered_folder_list_RTst)
uid2dose = build_uid_index_for_rtdose(filtered_folder_list_RTDOSE)

print(f"[Index] RTST mapped: {len(uid2rtst)}, RTDOSE mapped: {len(uid2dose)}")


[변경 전 흐름]

CT Load

ct_resample_crop_center (Center 기준 리샘플)

RTST Masking

Dose Load & Resample


[변경 후 흐름 - 실험용]

CT Load

Dose Load (위치 이동)

ct_resample_align_to_dose (Dose 기준 리샘플)

RTST Masking

Dose Resample

In [ ]:
# =========================================================
# [새로 추가된 함수] Dose 좌표계 기준 CT 리샘플링
# =========================================================
def ct_resample_align_to_dose(
    input_image: sitk.Image,   # CT Image (환자 위치 파악용)
    dose_image: sitk.Image,    # Dose Image (격자 기준점 역할)
    new_size=(256, 256, 256),
    new_spacing=(2.0, 2.0, 2.0)
):
    """
    Dose의 Grid 위상(Phase)을 유지하면서, CT 영상이 중앙에 오도록 
    '정수배 이동(Integer Snapping)'을 수행하는 리샘플링 함수입니다.
    """
    print("ct_resample_align_to_dose (Integer Snapping) start")

    # 1. CT의 물리적 중심(Center) 계산 (환자가 어디 있는지 알기 위해)
    ct_size = np.array(input_image.GetSize())
    ct_spacing = np.array(input_image.GetSpacing())
    ct_origin = np.array(input_image.GetOrigin())
    ct_direction = np.array(input_image.GetDirection()).reshape(3,3)
    
    ct_fov = ct_size * ct_spacing
    half_ct_fov = ct_direction @ (0.5 * ct_fov)
    ct_center = ct_origin + half_ct_fov
    
    # 2. Dose의 Origin (절대 기준점)
    dose_origin = np.array(dose_image.GetOrigin())
    
    # 3. [핵심] 이상적인 시작점(Ideal Origin) 계산
    #    (그냥 중앙 정렬을 한다면 시작했어야 할 위치)
    final_new_size = np.array(new_size)
    np_new_spacing = np.array(new_spacing)
    
    new_fov = final_new_size * np_new_spacing
    half_new_fov = ct_direction @ (0.5 * new_fov)
    ideal_origin = ct_center - half_new_fov
    
    # 4. [핵심] "Dose Origin" 기준으로 정수배 보정 (Snapping)
    #    이상적인 위치와 Dose Origin 사이의 거리를 구함
    offset = ideal_origin - dose_origin
    
    #    거리를 격자 간격(Spacing)으로 나누어 반올림하고 다시 곱함 -> 정수배가 됨
    #    예: 차이가 10.3mm, 간격 2mm -> 5칸(10mm)으로 보정 (0.3mm 버림)
    snapped_offset = np.round(offset / np_new_spacing) * np_new_spacing
    
    #    최종 Origin = Dose Origin + 정수배 Offset
    #    이렇게 하면 격자 눈금은 Dose와 완벽히 일치(Phase Lock)하면서 환자는 중앙에 옴
    final_origin = (dose_origin + snapped_offset).tolist()
    
    # 5. 리샘플링 수행
    resample_filter = sitk.ResampleImageFilter()
    resample_filter.SetSize(list(new_size))
    resample_filter.SetOutputSpacing(list(new_spacing))
    resample_filter.SetOutputOrigin(final_origin)     # 보정된 Origin 사용
    resample_filter.SetOutputDirection(input_image.GetDirection()) # 방향은 CT/Dose 동일 가정
    resample_filter.SetInterpolator(sitk.sitkLinear)
    resample_filter.SetDefaultPixelValue(-1000) # CT Air
    
    output_image = resample_filter.Execute(input_image)
    
    # 6. 좌표 계산 (리턴용)
    rx = final_origin[0] + np.arange(new_size[0]) * new_spacing[0]
    ry = final_origin[1] + np.arange(new_size[1]) * new_spacing[1]
    rz = final_origin[2] + np.arange(new_size[2]) * new_spacing[2]

    return output_image, rx, ry, rz
# =========================================================
# [수정된 함수] process_one_ct_folder
# =========================================================
def process_one_ct_folder(index):
    """
    index: CT DICOM folder path
    return: (index, study_id, ok, message_or_outpath)
    """
    try:
        # ✅ 스레드 병렬 + SITK 내부 멀티스레드가 겹치면 느려질 수 있어 1로 제한 권장
        sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)

        study_id = ""
        ct_study_uid = None

        print("-" * 80)
        print("[CASE]", index)

        # --- 1. CT read ---
        series_reader = sitk.ImageSeriesReader()
        dicom_series = series_reader.GetGDCMSeriesFileNames(index)
        series_reader.SetFileNames(dicom_series)
        series_reader.LoadPrivateTagsOn()
        ct_image = series_reader.Execute()

        print("Original Image Size:", ct_image.GetSize())
        print("Original Spacing:", ct_image.GetSpacing())
        print("Original Origin:", ct_image.GetOrigin())

        # --- CT UID / StudyID 읽기 (Metadata only) ---
        for ct_dicom_file in sorted(os.listdir(index)):
            if not ct_dicom_file.endswith(".dcm"):
                continue
            ct_ds = pydicom.dcmread(os.path.join(index, ct_dicom_file), stop_before_pixels=True, force=True)
            if getattr(ct_ds, "InstanceNumber", None) == 1:
                study_id = getattr(ct_ds, "StudyID", "")
                ct_study_uid = getattr(ct_ds, "StudyInstanceUID", None)
                rescale_slope = getattr(ct_ds, "RescaleSlope", 1)
                rescale_intercept = getattr(ct_ds, "RescaleIntercept", 0)
                print("CT StudyID:", study_id)
                print("CT StudyInstanceUID:", ct_study_uid)
                print("RescaleSlope:", rescale_slope, "RescaleIntercept:", rescale_intercept)
                break

        if ct_study_uid is None:
            return (index, study_id, False, "CT StudyInstanceUID not found")

        # =========================================================================
        # [수정] 2. RTDOSE를 먼저 로드하여 Grid Reference로 사용
        # =========================================================================
        rtdose_path = uid2dose.get(ct_study_uid, None)
        if rtdose_path is None:
            return (index, study_id, False, f"RTDOSE not found for UID: {ct_study_uid}")
        
        print(f"found RTDOSE (for Grid Alignment): {rtdose_path}")
        rtdose_image_raw = sitk.ReadImage(rtdose_path)

        # =========================================================================
        # [수정] 3. CT Resampling (Dose Origin 기준 정렬)
        # =========================================================================
        # 기존 ct_resample_crop_center 대신 새로 만든 함수 사용
        resampled_ct, x_c, y_c, z_c = ct_resample_align_to_dose(
            ct_image,
            rtdose_image_raw,  # Dose 이미지를 기준으로 넘겨줌
            new_size=(target_x_axis, target_y_axis, target_z_axis), # 512 or 256
            new_spacing=new_spacing # 1.0 or 2.0
        )

        print("Output size:", resampled_ct.GetSize())
        print("Output spacing:", resampled_ct.GetSpacing())
        print("Output origin:", resampled_ct.GetOrigin())

        resampled_ct_image = sitk.GetArrayFromImage(resampled_ct)  # (z,y,x)
        resampled_ct_image = resampled_ct_image.transpose(2, 1, 0)   # (x,y,z)
        print("resampled_ct_image size:", resampled_ct_image.shape)

        # =========================================================================
        # 4. RTST 매칭 + plastimatch convert + Mask Resampling
        # =========================================================================
        RTST_file = uid2rtst.get(ct_study_uid, None)
        if RTST_file is None:
            return (index, study_id, False, f"RTST not found for UID: {ct_study_uid}")

        RTST_path = os.path.dirname(RTST_file)
        print(f"found RTST: {RTST_file}")

        rt_st_folder_path = os.path.join(
            "mha_folder",
            os.path.basename(os.path.dirname(RTST_path)),
            f"{'_'.join(os.path.basename(RTST_path).split('_')[0:5])}_{ct_study_uid.replace('.', '_')}"
        )
        os.makedirs(rt_st_folder_path, exist_ok=True)
        print("mask out folder:", rt_st_folder_path)

        # --- convert 캐시 확인 ---
        need = set(selected_indices)
        exist = set(os.listdir(rt_st_folder_path))
        if not need.issubset(exist):
            # Plastimatch는 원본 CT를 참조하여 MHA 생성 (좌표계는 원본 CT 기준)
            command = [
                "plastimatch", "convert",
                "--input", RTST_file,
                "--referenced-ct", index,
                "--output-prefix", rt_st_folder_path
            ]
            subprocess.run(command, check=True, text=True, capture_output=True)
            print("Conversion successful")
        else:
            print("Conversion skipped (cached MHA exists)")

        mha_file_list = os.listdir(rt_st_folder_path)

        # ROI 선택/정렬
        filtered_files = [f for f in mha_file_list if f in selected_indices]
        if not all(x in filtered_files for x in selected_indices):
            failed_items = [x for x in desired_order if x not in filtered_files]
            return (index, study_id, False, f"Missing ROI MHA: {failed_items}")

        sorted_results = [f for f in desired_order if f in filtered_files]

        # --- RTST resample to resampled_ct (NN) ---
        # 중요: resampled_ct가 이제 Dose Grid를 따르므로, 마스크도 Dose Grid로 리샘플됨
        rtst_image_slices = np.zeros((target_x_axis, target_y_axis, resampled_ct.GetSize()[2], 0), dtype=np.uint8)

        for mha_name in sorted_results:
            mha_file = os.path.join(rt_st_folder_path, mha_name)
            mask_img = sitk.ReadImage(mha_file)
            
            # 여기서 resampled_ct(Dose Grid)로 변환됨 -> Partial Volume 효과가 Matlab과 유사해짐
            mask_rs = resample_mask_to_reference(mask_img, resampled_ct)  # NN interpolation

            mask_arr = sitk.GetArrayFromImage(mask_rs).astype(np.uint8)  # (z,y,x)
            mask_arr = mask_arr.transpose(2, 1, 0)                         # (x,y,z)
            mask_arr = mask_arr[..., None]
            rtst_image_slices = np.concatenate((rtst_image_slices, mask_arr), axis=-1)

        print("rtst_image_slices:", rtst_image_slices.shape)

        # =========================================================================
        # 5. Dose Scaling + Resample + Norm
        # =========================================================================
        ds_rtdose = pydicom.dcmread(rtdose_path, force=True)
        
        # DoseGridScaling 적용 (Gy)
        scaling = float(ds_rtdose.DoseGridScaling)
        rtdose_img_gy = sitk.Cast(rtdose_image_raw, sitk.sitkFloat32) * scaling

        # Dose Resample (Target: resampled_ct)
        # resampled_ct가 이미 Dose의 Origin/Direction을 가지고 있으므로,
        # 여기서는 Spacing 차이(3mm -> 2mm)에 대한 보간만 수행됨. 
        # 실험용으로 'linear' 추천 (bspline은 오버슈팅 가능성 있음)
        dose_rs = resample_dose_to_reference(rtdose_img_gy, resampled_ct, interp="linear", default_value=0.0)
        
        dose_rs_arr = sitk.GetArrayFromImage(dose_rs).astype(np.float32)   # (z,y,x)
        dose_rs_arr = dose_rs_arr.transpose(2, 1, 0)                         # (x,y,z)
        print("dose_rs_arr max (Gy):", float(dose_rs_arr.max()))

        # Normalization
        dose_norm = (dose_rs_arr / target_value).astype(np.float32)[..., None]
        print("dose_norm max:", float(dose_norm.max()))

        # =========================================================================
        # 6. Origin DVH Calculation (Optional)
        # =========================================================================
        case_id = index.split("/")[2]
        out_csv = os.path.join(output_dir, "CSV", case_id, "origin_DICOM_DVH.csv")
        os.makedirs(os.path.dirname(out_csv), exist_ok=True)

        roi_name_map = {
            "CTV": "CTV_WB", "Heart": "Heart", "Ipsi_Lung": "Ipsi_Lung",
            "Contra_Lung": "Contra_Lung", "Contra_Breast": "Contra_Breast",
            "A_LAD": "A_LAD", "External": "External",
        }

        if not os.path.exists(out_csv):
            ct_files = sorted([os.path.join(index, f) for f in os.listdir(index) if f.endswith(".dcm")])
            ct_dcm_list = [pydicom.dcmread(f) for f in ct_files]

            compute_and_save_origin_dicom_dvh(
                out_csv_path=out_csv,
                study_id=study_id,
                case_id=case_id,
                rtstruct_path=RTST_file,
                ds_rtdose=ds_rtdose,
                ct_dcm_list=ct_dcm_list,
                prescription_gy=prescription,
                roi_name_map=roi_name_map,
                bin_width_gy=0.1,
                include_full_dvh=True
            )
            print("[origin DVH] saved ->", out_csv)
        else:
            print("[origin DVH] exists -> skip")

        # =========================================================================
        # 7. NPY Assemble & Save
        # =========================================================================
        ct_ch = resampled_ct_image[..., None].astype(np.float32)  # (x,y,z,1)
        result_npy = np.concatenate((ct_ch, rtst_image_slices, dose_norm), axis=-1)
        print("result_npy:", result_npy.shape)

        # Rotate / Flip
        result_npy = np.rot90(result_npy, 3, axes=(0, 1))
        result_npy = np.flip(result_npy, axis=1)

        # Save
        save_dir = os.path.join(output_dir, "NPY", index.split("/")[1])
        os.makedirs(save_dir, exist_ok=True)
        # 파일명에 origin 정보를 포함하여 실험 결과 구별
        out_name = f"{study_id}_{index.split('/')[2]}_dose_aligned.npy"
        save_path = os.path.join(save_dir, out_name)
        np.save(save_path, result_npy)
        print("save done:", save_path)

        # DVH quick check
        load_dvh_npy(save_path)

        return (index, study_id, True, save_path)

    except Exception as e:
        msg = f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
        return (index, study_id, False, msg)

In [ ]:
# # =========================
# # 1) CT 폴더 하나 처리 함수 (Thread에서 실행)
# # =========================
# def process_one_ct_folder(index):
#     """
#     index: CT DICOM folder path
#     return: (index, ok, message_or_outpath)
#     """
#     try:
#         # ✅ 스레드 병렬 + SITK 내부 멀티스레드가 겹치면 느려질 수 있어 1로 제한 권장
#         sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)

#         study_id = ""
#         ct_study_uid = None

#         print("-"*80)
#         print("[CASE]", index)

#         # --- CT read ---
#         series_reader = sitk.ImageSeriesReader()
#         dicom_series = series_reader.GetGDCMSeriesFileNames(index)
#         series_reader.SetFileNames(dicom_series)
#         series_reader.LoadPrivateTagsOn()
#         ct_image = series_reader.Execute()

#         print("Original Image Size:", ct_image.GetSize())
#         print("Original Spacing:", ct_image.GetSpacing())
#         print("Original Origin:", ct_image.GetOrigin())

#         # --- CT UID / StudyID 읽기 (가볍게: stop_before_pixels) ---
#         # 기존처럼 InstanceNumber==1 찾되, 픽셀은 안 읽게 해서 빠르게
#         for ct_dicom_file in sorted(os.listdir(index)):
#             if not ct_dicom_file.endswith(".dcm"):
#                 continue
#             ct_ds = pydicom.dcmread(os.path.join(index, ct_dicom_file), stop_before_pixels=True, force=True)
#             if getattr(ct_ds, "InstanceNumber", None) == 1:
#                 study_id = getattr(ct_ds, "StudyID", "")
#                 ct_study_uid = getattr(ct_ds, "StudyInstanceUID", None)
#                 rescale_slope = getattr(ct_ds, "RescaleSlope", 1)
#                 rescale_intercept = getattr(ct_ds, "RescaleIntercept", 0)
#                 print("CT StudyID:", study_id)
#                 print("CT StudyInstanceUID:", ct_study_uid)
#                 print("RescaleSlope:", rescale_slope, "RescaleIntercept:", rescale_intercept)
#                 break

#         if ct_study_uid is None:
#             return (index, False, "CT StudyInstanceUID not found")

#         # --- CT resample ---
#         # process_one_ct_folder 내부 수정 예시
#         resampled_ct, x_sample_coords, y_sample_coords, z_sample_coords = ct_resample_crop_center(
#             ct_image,
#             new_size=(target_x_axis, target_y_axis, target_z_axis),
#             new_spacing=new_spacing,
#             use_center_crop=False  # <--- 이 부분을 False로 설정하여 MATLAB과 정렬을 맞춤
#         )

#         print("Output size:", resampled_ct.GetSize())
#         print("Output spacing:", resampled_ct.GetSpacing())
#         print("Output origin:", resampled_ct.GetOrigin())

#         resampled_ct_image = sitk.GetArrayFromImage(resampled_ct)  # (z,y,x)
#         resampled_ct_image = resampled_ct_image.transpose(2,1,0)   # (x,y,z)
#         print("resampled_ct_image size:", resampled_ct_image.shape)

#         # =========================
#         # 2) RTST 매칭 + plastimatch convert
#         # =========================
#         RTST_file = uid2rtst.get(ct_study_uid, None)
#         if RTST_file is None:
#             return (index, False, f"RTST not found for UID: {ct_study_uid}")

#         RTST_path = os.path.dirname(RTST_file)
#         print(f"found RTST: {RTST_file}")

#         # ✅ 폴더명이 케이스별로 유일하도록 UID를 포함시키는 걸 권장(동시 실행 충돌 방지)
#         rt_st_folder_path = os.path.join(
#             "mha_folder",
#             os.path.basename(os.path.dirname(RTST_path)),
#             f"{'_'.join(os.path.basename(RTST_path).split('_')[0:5])}_{ct_study_uid.replace('.', '_')}"
#         )
#         os.makedirs(rt_st_folder_path, exist_ok=True)
#         print("mask out folder:", rt_st_folder_path)

#         # --- convert 캐시: 필요한 ROI 파일이 다 있으면 convert 스킵 ---
#         need = set(selected_indices)
#         exist = set(os.listdir(rt_st_folder_path))
#         if not need.issubset(exist):
#             command = [
#                 "plastimatch", "convert",
#                 "--input", RTST_file,
#                 "--referenced-ct", index,
#                 "--output-prefix", rt_st_folder_path
#             ]
#             subprocess.run(command, check=True, text=True, capture_output=True)
#             print("Conversion successful")
#         else:
#             print("Conversion skipped (cached MHA exists)")

#         mha_file_list = os.listdir(rt_st_folder_path)
#         print("mha_file_list:", mha_file_list)

#         # (디버그) mask / ct geometry 확인
#         tmp = sitk.ReadImage(os.path.join(rt_st_folder_path, sorted(mha_file_list)[0]))
#         print("mask spacing/origin/direction:", tmp.GetSpacing(), tmp.GetOrigin(), tmp.GetDirection())
#         print("ct   spacing/origin/direction:", ct_image.GetSpacing(), ct_image.GetOrigin(), ct_image.GetDirection())

#         # ROI 선택/정렬
#         filtered_files = [f for f in mha_file_list if f in selected_indices]
#         if not all(x in filtered_files for x in selected_indices):
#             failed_items = [x for x in desired_order if x not in filtered_files]
#             return (index, False, f"Missing ROI MHA: {failed_items}")

#         sorted_results = [f for f in desired_order if f in filtered_files]

#         # --- RTST resample to resampled_ct (NN) ---
#         # ⚠️ 여기 버그 주의: (target_x_axis, target_y_axis, z, 0)로 잡아야 함 (y를 x로 쓰면 틀어짐)
#         rtst_image_slices = np.zeros((target_x_axis, target_y_axis, resampled_ct.GetSize()[2], 0), dtype=np.uint8)

#         for mha_name in sorted_results:
#             mha_file = os.path.join(rt_st_folder_path, mha_name)
#             mask_img = sitk.ReadImage(mha_file)
#             mask_rs = resample_mask_to_reference(mask_img, resampled_ct)  # NN

#             mask_arr = sitk.GetArrayFromImage(mask_rs).astype(np.uint8)  # (z,y,x)
#             mask_arr = mask_arr.transpose(2,1,0)                         # (x,y,z)
#             mask_arr = mask_arr[..., None]
#             rtst_image_slices = np.concatenate((rtst_image_slices, mask_arr), axis=-1)

#         print("rtst_image_slices:", rtst_image_slices.shape)

#         # =========================
#         # 3) RTDOSE 매칭 + dose scaling + bspline resample + norm
#         # =========================
#         rtdose_path = uid2dose.get(ct_study_uid, None)
#         if rtdose_path is None:
#             return (index, False, f"RTDOSE not found for UID: {ct_study_uid}")

#         ds_rtdose = pydicom.dcmread(rtdose_path, force=True)
#         print(f"found RTDOSE: {rtdose_path}")

#         rtdose_image = sitk.ReadImage(rtdose_path)

#         # DoseGridScaling 적용 후 (Gy)
#         scaling = float(ds_rtdose.DoseGridScaling)
#         rtdose_img_gy = sitk.Cast(rtdose_image, sitk.sitkFloat32) * scaling

#         # bspline resample to CT grid
#         dose_rs = resample_dose_to_reference(rtdose_img_gy, resampled_ct, interp="bspline", default_value=0.0)
#         dose_rs_arr = sitk.GetArrayFromImage(dose_rs).astype(np.float32)   # (z,y,x)
#         dose_rs_arr = dose_rs_arr.transpose(2,1,0)                         # (x,y,z)
#         print("dose_rs_arr max (Gy):", float(dose_rs_arr.max()))

#         # normalization (0~1)
#         dose_norm = (dose_rs_arr / target_value).astype(np.float32)[..., None]
#         print("dose_norm max:", float(dose_norm.max()))

#         # =========================
#         # 4) origin DVH (원하면 스킵/캐시 가능)
#         # =========================
#         case_id = index.split("/")[2]
#         out_csv = os.path.join(output_dir, "CSV", case_id, "origin_DICOM_DVH.csv")
#         os.makedirs(os.path.dirname(out_csv), exist_ok=True)

#         roi_name_map = {
#             "CTV": "CTV_WB",
#             "Heart": "Heart",
#             "Ipsi_Lung": "Ipsi_Lung",
#             "Contra_Lung": "Contra_Lung",
#             "Contra_Breast": "Contra_Breast",
#             "A_LAD": "A_LAD",
#             "External": "External",
#         }

#         # ✅ 파일 많으면 DVH 계산이 또 무거울 수 있으니, 이미 있으면 스킵(선택)
#         if not os.path.exists(out_csv):
#             ct_files = sorted([os.path.join(index, f) for f in os.listdir(index) if f.endswith(".dcm")])
#             ct_dcm_list = [pydicom.dcmread(f) for f in ct_files]

#             compute_and_save_origin_dicom_dvh(
#                 out_csv_path=out_csv,
#                 study_id=study_id,
#                 case_id=case_id,
#                 rtstruct_path=RTST_file,
#                 ds_rtdose=ds_rtdose,
#                 ct_dcm_list=ct_dcm_list,
#                 prescription_gy=prescription,
#                 roi_name_map=roi_name_map,
#                 bin_width_gy=0.1,
#                 include_full_dvh=True
#             )
#             print("[origin DVH] saved ->", out_csv)
#         else:
#             print("[origin DVH] exists -> skip")

#         # =========================
#         # 5) NPY assemble + rotate/flip (전체 채널 동일 적용)
#         # =========================
#         ct_ch = resampled_ct_image[..., None].astype(np.float32)  # (x,y,z,1)
#         result_npy = np.concatenate((ct_ch, rtst_image_slices, dose_norm), axis=-1)
#         print("result_npy:", result_npy.shape)

#         # 전체 채널 동일 rotate/flip
#         result_npy = np.rot90(result_npy, 3, axes=(0,1))
#         result_npy = np.flip(result_npy, axis=1)

#         # save
#         save_dir = os.path.join(output_dir, "NPY", index.split("/")[1])
#         os.makedirs(save_dir, exist_ok=True)
#         out_name = f"{study_id}_{index.split('/')[2]}_origin_{resampled_ct.GetOrigin()[0]}_{resampled_ct.GetOrigin()[1]}_{resampled_ct.GetOrigin()[2]}.npy"
#         save_path = os.path.join(save_dir, out_name)
#         np.save(save_path, result_npy)
#         print("save done:", save_path)

#         # DVH quick check
#         load_dvh_npy(save_path)

#         # return (index, True, save_path)
#         return (index, study_id, True, save_path)

#     except Exception as e:
#         msg = f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
#         # return (index, False, msg)
#         return (index, study_id, False, msg)

In [ ]:
# =========================
# 2) 멀티쓰레드 실행
# =========================
max_workers = 8
results = []

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(process_one_ct_folder, idx) for idx in filtered_folder_list_CT]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Processing cases"):
        results.append(fut.result())

ok  = [r for r in results if r[2]]      # (index, study_id, ok, msg)
bad = [r for r in results if not r[2]]

print("\n==== Summary ====")
print("OK :", len(ok))
print("BAD:", len(bad))

if bad:
    # 예시 1개 출력 (study_id 포함)
    idx, sid, _, msg = bad[0]
    print(f"\nExample failure: CT StudyID: {sid} | CT folder: {idx}\n{msg}")


In [ ]:
NPY_dir = os.path.join(output_dir, "NPY/") 
NPY_file_list = []
folder_list = os.listdir(NPY_dir)
NPY_file_list = [NPY_dir + index + "/" + j for index in folder_list for j in os.listdir(NPY_dir + index)]
print(len(NPY_file_list))

## clinical goal check data pass

dvh 검사후 치료 조건을 만족하는 case 및 만족하지 못하는 case 추출

In [ ]:
def v_at_step_1d(dose_vals: np.ndarray, thr_gy: float) -> float:
    """계단형 Vx: mean(dose>=thr)*100"""
    if dose_vals.size == 0:
        return float("nan")
    return float((dose_vals >= thr_gy).mean() * 100.0)

def v_at_matlablike_1d(dose_vals: np.ndarray, thr_gy: float) -> float:
    """
    MATLAB Step_E 근사:
    - dose sort(desc)
    - acc_percent = (1..N)/N*100
    - unique(dose) + 해당 index의 acc_percent
    - interp1(dose_unique, acc_percent_unique, thr, 'linear','extrap') 후 0~100 clamp
    """
    if dose_vals.size == 0:
        return float("nan")

    s = np.sort(dose_vals)[::-1]   # desc
    n = s.size
    acc_pct = (np.arange(1, n + 1, dtype=np.float32) / n) * 100.0

    # MATLAB: [dose_unique, index] = unique(dose_interp_sort);
    # Python: np.unique는 u를 오름차순으로 반환함
    u, idx = np.unique(s, return_index=True)   # u: asc
    acc_u = acc_pct[idx]

    # u asc에 맞춰 정렬(안전)
    order = np.argsort(u)
    u = u[order]
    acc_u = acc_u[order]

    f = interp1d(u, acc_u, kind="linear", fill_value="extrapolate", assume_sorted=True)
    v = float(f(thr_gy))
    return float(np.clip(v, 0.0, 100.0))

In [ ]:
CHANNEL_NAMES = [
    'CT',                # idx 0
    'A_LAD.mha',         # idx 1
    'CTV.mha',           # idx 2
    'Contra_breast.mha', # idx 3
    'External.mha',      # idx 4
    'Heart.mha',         # idx 5
    'Ipsi_Lung.mha',     # idx 6
    'Contra_Lung.mha',   # idx 7
    'RING.mha',          # idx 8
    'RTDOSE'             # idx 9
]

# 인덱스 매핑
IDX_CTV = 2
IDX_HEART = 5
IDX_IPSI_LUNG = 6
IDX_CONTRA_LUNG = 7
IDX_CONTRA_BREAST = 3
IDX_DOSE = 9

# 선량 평가 기준
PRESCRIPTION_GY = 26.0
V95_THRESHOLD_GY = PRESCRIPTION_GY * 0.95  # 24.7 Gy
NORM_FACTOR = PRESCRIPTION_GY * 1.1        # 28.6 Gy (복원용)


# 2mm voxel 안에서 1mm sub-voxel center 8개 (±0.5mm) → index로는 ±0.25 (2mm spacing 가정)
_SUB_OFFSETS = np.array([
    [-0.25, -0.25, -0.25],
    [-0.25, -0.25, +0.25],
    [-0.25, +0.25, -0.25],
    [-0.25, +0.25, +0.25],
    [+0.25, -0.25, -0.25],
    [+0.25, -0.25, +0.25],
    [+0.25, +0.25, -0.25],
    [+0.25, +0.25, +0.25],
], dtype=np.float32)

def sample_dose_1mm_like_from_2mm(dose_xyz: np.ndarray,
                                 mask_xyz: np.ndarray,
                                 thr: float = 0.5,
                                 chunk_vox: int = 200_000):
    """
    dose_xyz: (X,Y,Z) 선량 (NPY 저장 축과 동일)
    mask_xyz: (X,Y,Z) ROI mask
    """
    mask_bin = (mask_xyz >= thr)
    idx = np.argwhere(mask_bin)  # (N,3) in (x,y,z)
    if idx.size == 0:
        return np.array([], dtype=np.float32)

    out = []
    for s in range(0, len(idx), chunk_vox):
        chunk = idx[s:s+chunk_vox].astype(np.float32)     # (n,3)
        coords = chunk[:, None, :] + _SUB_OFFSETS[None,:,:]  # (n,8,3)
        coords = coords.reshape(-1, 3).T                  # (3, n*8) = (x,y,z)

        vals = map_coordinates(dose_xyz, coords, order=1, mode="constant", cval=0.0)
        out.append(vals.astype(np.float32))

    return np.concatenate(out, axis=0)

# ==============================================================================
# 2. [핵심] Hybrid Metric 계산 함수
# ==============================================================================

# def calculate_metric_clinical(
#     dose_gy,
#     mask_soft,
#     metric_type,
#     value_threshold=None,
#     mode="2mm",      # "2mm" or "1mm_like"
#     thr=0.5
# ):
#     # 1) MATLAB과 동일한 inside 정의
#     mask_bin = (mask_soft >= thr)

#     # 2) empty check도 mask_bin 기준으로
#     if not np.any(mask_bin):
#         return 0.0

#     # 3) sampled 생성 (여기가 MATLAB 정합의 핵심)
#     if mode == "2mm":
#         # dose_gy와 mask_soft가 같은 grid(2mm)라면 MATLAB UseInterpolation=0과 유사
#         sampled = dose_gy[mask_bin]
#     elif mode == "1mm_like":
#         # 반드시 mask_soft가 아니라 mask_bin을 넘기세요.
#         sampled = sample_dose_1mm_like_from_2mm(dose_gy, mask_bin)  # <= 여기 수정
#     else:
#         raise ValueError("mode must be '2mm' or '1mm_like'")

#     if sampled.size == 0:
#         return 0.0

#     # 4) metric 계산 (이 부분은 지금 코드 그대로 OK)
#     if metric_type == "V_x":
#         return 100.0 * (sampled >= value_threshold).mean()
#     elif metric_type == "D_mean":
#         return float(np.mean(sampled))
#     elif metric_type == "D_max":
#         return float(np.max(sampled))
#     elif metric_type == "D_min":
#         return float(np.min(sampled))
#     elif metric_type == "D_100":
#         return float(np.min(sampled))

#     return 0.0

# new version - for matlab
def calculate_metric_clinical(
    dose_gy,
    mask_soft,
    metric_type,
    value_threshold=None,
    mode="2mm",
    thr=0.5,
    vx_method="step",   # <-- 추가: "step" or "matlablike"
):
    mask_bin = (mask_soft >= thr)
    if not np.any(mask_bin):
        return 0.0

    if mode == "2mm":
        sampled = dose_gy[mask_bin]
    elif mode == "1mm_like":
        sampled = sample_dose_1mm_like_from_2mm(dose_gy, mask_soft, thr=thr)  # 지금 방식 유지해도 OK
        # (더 최적화하려면 아래 3) 참고)
    else:
        raise ValueError("mode must be '2mm' or '1mm_like'")

    if sampled.size == 0:
        return 0.0

    if metric_type == "V_x":
        if vx_method == "step":
            return v_at_step_1d(sampled, value_threshold)
        elif vx_method == "matlablike":
            return v_at_matlablike_1d(sampled, value_threshold)
        else:
            raise ValueError("vx_method must be 'step' or 'matlablike'")

    elif metric_type == "D_mean":
        return float(np.mean(sampled))
    elif metric_type == "D_max":
        return float(np.max(sampled))
    elif metric_type == "D_min":
        return float(np.min(sampled))
    elif metric_type == "D_100":
        return float(np.min(sampled))

    return 0.0

# ==============================================================================
# 3. 메인 분석 함수
# ==============================================================================

def analyze_dvh_fixed(output_dir, npy_folder_name="NPY"):
    NPY_dir = os.path.join(output_dir, npy_folder_name)
    
    NPY_file_list = []
    if not os.path.exists(NPY_dir):
        print(f"Error: 경로가 없습니다 -> {NPY_dir}")
        return

    for root, dirs, files in os.walk(NPY_dir):
        for file in files:
            if file.endswith('.npy'):
                NPY_file_list.append(os.path.join(root, file))

    print(f"Total NPY Files found: {len(NPY_file_list)}")
    
    results = []

    for npy_path in tqdm(NPY_file_list, desc="Analyzing (Hybrid Mode)"):
        try:
            # 1. 데이터 로드
            data = np.load(npy_path)
            case_id = npy_path.split(os.sep)[-2]
            file_name = os.path.basename(npy_path)
            
            # 채널 체크
            if data.shape[-1] != len(CHANNEL_NAMES):
                continue

            # 2. Dose 복원
            dose_norm = data[..., IDX_DOSE].astype(np.float32)
            dose_gy = dose_norm * NORM_FACTOR  
            
            # 3. 지표 계산 (새로운 함수 사용)
            case_res = {
                "case_id": case_id, 
                "Pass_All": True, 
                "Fail_Reasons": []
            }
            
            # (A) CTV
            mask_ctv = data[..., IDX_CTV]
            if np.sum(mask_ctv) > 0:
                # --- V95: step vs matlab-like (실험 2) ---
                v95_step = calculate_metric_clinical(
                    dose_gy, mask_ctv, "V_x", V95_THRESHOLD_GY,
                    mode="2mm", vx_method="step"
                )
                v95_mat = calculate_metric_clinical(
                    dose_gy, mask_ctv, "V_x", V95_THRESHOLD_GY,
                    mode="2mm", vx_method="matlablike"
                )

                # V95 -> Binary로 계산됨 (MIM 일치율 상승)
                # v95 = calculate_metric_clinical(dose_gy, mask_ctv, "V_x", V95_THRESHOLD_GY)
                v95 = v95_step
                # Dmax -> Soft로 계산됨 (정밀도 유지)
                dmax = calculate_metric_clinical(dose_gy, mask_ctv, "D_max")

                # ✅ 추가 0이 나오면 문제 있음 확인용
                dmin  = calculate_metric_clinical(dose_gy, mask_ctv, "D_min")
                d100  = calculate_metric_clinical(dose_gy, mask_ctv, "D_100")
                
                # case_res["CTV_V95(%)"] = v95
                case_res["CTV_V95_step(%)"] = v95_step
                case_res["CTV_V95_matlabLike(%)"] = v95_mat
                case_res["CTV_V95_step_minus_matlab(%p)"] = v95_step - v95_mat


                # 기존 컬럼 유지(원하면)
                case_res["CTV_V95(%)"] = v95_step
                case_res["CTV_Dmax(Gy)"] = dmax
                case_res["CTV_Dmin(Gy)"]   = dmin
                case_res["CTV_D100(Gy)"]   = d100
                
                if v95 < 95.0:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("CTV_V95<95")
                if (dmax / PRESCRIPTION_GY) > 1.07:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("CTV_Vmax>1.07")
                
                # ✅ 0 Gy(또는 매우 작은 값) 의심 탐지: DICOM 변환/정합 문제 후보
                # (완전 0이 아니어도 거의 0이면 suspicious)
                if dmin <= 0.05:  # 0.05 Gy는 예시, 필요하면 0.1~0.2로 조정
                    case_res["Fail_Reasons"].append("CTV_Dmin≈0 (Check DICOM/ROI/Dose alignment)")

            else:
                case_res["CTV_V95(%)"] = 0.0
                case_res["CTV_V95_step(%)"] = 0.0
                case_res["CTV_V95_matlabLike(%)"] = 0.0
                case_res["CTV_V95_step_minus_matlab(%p)"] = 0.0
                case_res["CTV_Dmax(Gy)"] = 0.0
                case_res["CTV_Dmin(Gy)"] = 0.0   # ✅ 추가
                case_res["CTV_D100(Gy)"] = 0.0   # ✅ 추가  
                case_res["Fail_Reasons"].append("CTV_Mask_Empty")

            # (B) Heart
            mask_heart = data[..., IDX_HEART]
            if np.sum(mask_heart) > 0:
                v7 = calculate_metric_clinical(dose_gy, mask_heart, "V_x", 7.0)
                v1_5 = calculate_metric_clinical(dose_gy, mask_heart, "V_x", 1.5)
                
                case_res["Heart_V7(%)"] = v7
                case_res["Heart_V1.5(%)"] = v1_5
                
                if v7 >= 3.0:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("Heart_V7>=3")
                if v1_5 >= 30.0:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("Heart_V1.5>=30")
            else:
                case_res["Heart_V7(%)"] = 0.0
                case_res["Heart_V1.5(%)"] = 0.0

            # (C) Ipsi Lung (V8 -> Binary)
            mask_ipsi = data[..., IDX_IPSI_LUNG]
            if np.sum(mask_ipsi) > 0:
                v8 = calculate_metric_clinical(dose_gy, mask_ipsi, "V_x", 8.0)
                case_res["IpsiLung_V8(%)"] = v8
                
                if v8 >= 15.0:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("IpsiLung_V8>=15")
            else:
                 case_res["IpsiLung_V8(%)"] = 0.0

            # (D) Contra Lung (Dmean -> Soft)
            mask_contra = data[..., IDX_CONTRA_LUNG]
            if np.sum(mask_contra) > 0:
                dmean = calculate_metric_clinical(dose_gy, mask_contra, "D_mean")
                case_res["ContraLung_Dmean(Gy)"] = dmean
                
                if dmean >= 2.0:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("ContraLung_Dmean>=2")
            else:
                case_res["ContraLung_Dmean(Gy)"] = 0.0

            # (E) Contra Breast (Dmean -> Soft)
            mask_cb = data[..., IDX_CONTRA_BREAST]
            if np.sum(mask_cb) > 0:
                dmean = calculate_metric_clinical(dose_gy, mask_cb, "D_mean")
                case_res["ContraBreast_Dmean(Gy)"] = dmean
                
                if dmean > 3.0:
                    case_res["Pass_All"] = False
                    case_res["Fail_Reasons"].append("ContraBreast_Dmean>3")
            else:
                case_res["ContraBreast_Dmean(Gy)"] = 0.0

            # 정리
            case_res["Fail_Reasons"] = "; ".join(case_res["Fail_Reasons"]) if case_res["Fail_Reasons"] else "None"
            case_res["file_name"] = file_name
            results.append(case_res)

        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue

    # CSV 저장
    if results:
        df = pd.DataFrame(results)
        # cols_order = [
        #     "case_id", "Pass_All", "Fail_Reasons", 
        #     "CTV_V95(%)", "CTV_Dmax(Gy)", "Heart_V7(%)", "Heart_V1.5(%)",
        #     "IpsiLung_V8(%)", "ContraLung_Dmean(Gy)", "ContraBreast_Dmean(Gy)",
        #     "file_name"
        # ]
        cols_order = [
            "case_id", "Pass_All", "Fail_Reasons",

            "CTV_V95(%)",
            "CTV_V95_step(%)",
            "CTV_V95_matlabLike(%)",
            "CTV_V95_step_minus_matlab(%p)",

            "CTV_Dmax(Gy)", "CTV_Dmin(Gy)", "CTV_D100(Gy)",
            "Heart_V7(%)", "Heart_V1.5(%)",
            "IpsiLung_V8(%)", "ContraLung_Dmean(Gy)", "ContraBreast_Dmean(Gy)",
            "file_name"
        ]

        final_cols = [c for c in cols_order if c in df.columns]
        df = df[final_cols]
        
        save_path = os.path.join(output_dir, "DVH_Analysis_load_dvh_Matched_matlablike.csv")
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"Saved: {save_path}")
        
        # 합격률 출력
        pass_rate = (df['Pass_All'].sum() / len(df)) * 100
        print(f"New Pass Rate: {pass_rate:.2f}%")
        print(df.head())

In [ ]:
analyze_dvh_fixed(output_dir)

In [ ]:
import os
import pandas as pd
import numpy as np

# 1) CSV 로드
csv_path = os.path.join(output_dir, "DVH_Analysis_load_dvh_Matched_matlablike.csv")
df = pd.read_csv(csv_path)

# Pass_All이 문자열로 읽히는 경우 방지 ("True"/"False")
if df["Pass_All"].dtype == object:
    df["Pass_All"] = df["Pass_All"].astype(str).str.lower().map({"true": True, "false": False})

# 2) Fail만 필터
fail_df = df[df["Pass_All"] == False].copy()

# 3) Fail_Reasons 정리 (None/NaN 처리)
fail_df["Fail_Reasons"] = fail_df["Fail_Reasons"].fillna("None").astype(str)

# ---- (A) 이유 분해 + 카운트 ----
def split_reasons(s):
    if s.strip() in ["", "None", "nan"]:
        return []
    return [x.strip() for x in s.split(";") if x.strip()]

fail_df["Fail_List"] = fail_df["Fail_Reasons"].apply(split_reasons)
fail_df["Fail_Count"] = fail_df["Fail_List"].apply(len)

# 이유별 one-hot 컬럼 생성
all_reasons = sorted({r for lst in fail_df["Fail_List"] for r in lst})
for r in all_reasons:
    col = f"R__{r}"
    fail_df[col] = fail_df["Fail_List"].apply(lambda lst: int(r in lst))

# ---- (B) CTV Dmin/D100 0Gy 의심 플래그 ----
# (네가 추가해둔 컬럼명이 CSV에 들어있다는 가정. 없으면 자동 스킵됨)
EPS_GY = 0.05  # "거의 0Gy" 판단 기준 (원하면 0.1~0.2로 조절)

if "CTV_Dmin(Gy)" in fail_df.columns:
    fail_df["CTV_Dmin_Suspicious0"] = (fail_df["CTV_Dmin(Gy)"].fillna(0) <= EPS_GY)
else:
    fail_df["CTV_Dmin_Suspicious0"] = False

if "CTV_D100(Gy)" in fail_df.columns:
    fail_df["CTV_D100_Suspicious0"] = (fail_df["CTV_D100(Gy)"].fillna(0) <= EPS_GY)
else:
    fail_df["CTV_D100_Suspicious0"] = False

fail_df["CTV_MinDose_Suspicious0"] = fail_df["CTV_Dmin_Suspicious0"] | fail_df["CTV_D100_Suspicious0"]

# ---- (C) 엑셀에 저장할 컬럼 구성 (존재하는 컬럼만) ----
base_cols = [
    "case_id", "Fail_Reasons", "Fail_Count",
    "CTV_V95(%)", "CTV_Dmax(Gy)", "CTV_Dmin(Gy)", "CTV_D100(Gy)",
    "ContraBreast_Dmean(Gy)", "Heart_V1.5(%)", "ContraLung_Dmean(Gy)",
    "Heart_V7(%)", "IpsiLung_V8(%)",
    "CTV_MinDose_Suspicious0",
    "file_name"
]
cols_to_save = [c for c in base_cols if c in fail_df.columns]

fail_summary = fail_df[cols_to_save].copy()

# 보기좋은 정렬: (의심0Gy 우선) -> 실패개수 많은 순 -> 이유 -> 케이스ID
sort_cols = [c for c in ["CTV_MinDose_Suspicious0", "Fail_Count", "Fail_Reasons", "case_id"] if c in fail_summary.columns]
ascending = [False, False, True, True][:len(sort_cols)]
fail_summary = fail_summary.sort_values(by=sort_cols, ascending=ascending)

# ---- (D) 이유 요약 테이블 ----
reason_counts = (
    pd.Series([r for lst in fail_df["Fail_List"] for r in lst])
      .value_counts()
      .rename_axis("Fail_Reason")
      .reset_index(name="Count")
)

# ---- (E) 이유별 케이스 목록 (피벗 스타일) ----
# 각 이유별로 해당 케이스들 모아보기용 (가독성 좋음)
reason_case_rows = []
for r in all_reasons:
    sub = fail_df[fail_df[f"R__{r}"] == 1][["case_id", "file_name"]].copy()
    sub.insert(0, "Fail_Reason", r)
    reason_case_rows.append(sub)
reason_case_list = pd.concat(reason_case_rows, ignore_index=True) if reason_case_rows else pd.DataFrame(columns=["Fail_Reason","case_id","file_name"])

# 4) 엑셀 저장 (여러 시트)
excel_name = "DVH_Fail_Cases_Summary_load_dvh_npy.xlsx"  # 오타 ecxel_name -> excel_name
save_excel_path = os.path.join(output_dir, excel_name)

with pd.ExcelWriter(save_excel_path, engine="openpyxl") as writer:
    fail_summary.to_excel(writer, sheet_name="Fail_Cases", index=False)
    reason_counts.to_excel(writer, sheet_name="Fail_Reason_Counts", index=False)
    reason_case_list.to_excel(writer, sheet_name="Fail_Reason_CaseList", index=False)

print(f"총 {len(fail_summary)}개의 실패 케이스가 저장되었습니다.")
print(f"저장 경로: {save_excel_path}")

print("\n[미리보기]")
print(fail_summary.head().to_string(index=False))


In [ ]:
# 1. 전체 분석 결과 CSV 로드
# (앞서 생성한 'DVH_Analysis_MIM_Matched.csv' 경로를 지정하세요)
csv_path = os.path.join(output_dir,"DVH_Analysis_load_dvh_Matched_matlablike.csv")
df = pd.read_csv(csv_path)

# 2. 실패(Fail) 케이스만 필터링
fail_df = df[df["Pass_All"] == False].copy()

# 3. 엑셀에 저장할 주요 컬럼 선택
# (보고 싶으신 컬럼들을 순서대로 나열했습니다)
cols_to_save = [
    "case_id",               # 케이스 ID
    "Fail_Reasons",          # 실패 원인 (가장 중요)
    "CTV_V95(%)",            # CTV V95 (기준: >= 95%)
    "CTV_Dmax(Gy)",          # CTV Vmax (기준: <= 1.07 * Prescription)
    "ContraBreast_Dmean(Gy)",# 반대쪽 유방 (기준: <= 3 Gy)
    "Heart_V1.5(%)",         # 심장 V1.5 (기준: < 30%)
    "ContraLung_Dmean(Gy)",  # 반대쪽 폐 (기준: < 2 Gy)
    "Heart_V7(%)",           # 심장 V7
    "IpsiLung_V8(%)",        # 동측 폐 V8
    "file_name"              # 원본 파일명 (참고용)
]

# 해당 컬럼만 추출
fail_summary = fail_df[cols_to_save]

# 4. 보기 좋게 정렬 (실패 원인별 -> 케이스 ID별)
fail_summary = fail_summary.sort_values(by=["Fail_Reasons", "case_id"])

# 5. 엑셀 파일로 저장
ecxel_name = "DVH_Fail_Cases_Summary.xlsx"
save_excel_path = os.path.join(output_dir,ecxel_name)
fail_summary.to_excel(save_excel_path, index=False)

print(f"총 {len(fail_summary)}개의 실패 케이스가 저장되었습니다.")
print(f"저장 경로: {save_excel_path}")

# (참고) 상위 5개 미리보기 출력
print("\n[미리보기]")
print(fail_summary.head().to_string())

## npy image just spread
check image good

In [ ]:
# 데이터와 파일명
# 1) output_dir 아래의 모든 .npy 파일 경로 수집 (재귀)
npy_files = []
for root, _, files in os.walk(output_dir):
    for fn in files:
        if fn.lower().endswith(".npy"):
            npy_files.append(os.path.join(root, fn))

if not npy_files:
    raise FileNotFoundError(f"No .npy files found under: {output_dir}")

# 2) 랜덤 1개 선택
random_npy_path = random.choice(npy_files)
print("Selected npy:", random_npy_path)

# 3) 로드
data = np.load(random_npy_path)
print("Loaded shape:", data.shape, "path:", random_npy_path)
slice_part_name = ['CT', 'A_LAD.mha', 'CTV.mha', 'Contra_Breast.mha', 'External.mha', 'Heart.mha', 'Ipsi_Lung.mha', 'Contra_Lung.mha', 'RING.mha', 'RTDOSE']

# z 축을 기준으로 4개의 슬라이스를 일정한 간격으로 선택
z_slices = np.linspace(0, data.shape[2] - 1, 10, dtype=int)
print(data.shape) # 데이터의 전체 크기 출력
print(z_slices)

# 선택한 z 슬라이스에 대해 이미지를 시각화하고 저장
fig, axes = plt.subplots(len(z_slices), 10, figsize=(20, 20))

for i, z in enumerate(z_slices):
    for j in range(10):
        ax = axes[i, j] if len(z_slices) > 1 else axes[j]
        
        slice_data = data[:, :, z, j]

        if 1 <= j <= 9:  # 이진 이미지 처리
            scaled_slice = slice_data
            scaled_slice_preprocessing = (slice_data * 255).astype(np.uint8)
                
        else:  # 그레이스케일 이미지 처리
            normalized_slice = (slice_data - np.min(slice_data)) / (np.max(slice_data) - np.min(slice_data))
            scaled_slice = (normalized_slice * 255).astype(np.uint8)
        
        ax.imshow(scaled_slice, cmap='gray')
        ax.axis('off')
        if i == 0:
            ax.set_title(f'{slice_part_name[j]}_n={j}')
        if j == 0:
            ax.set_ylabel(f'z={z}')

plt.tight_layout()
plt.show()

## split Train Test 
0~8 input/ 9 output

In [ ]:
NPY_dir = NPY_dir 
NPY_file_list = []
folder_list = os.listdir(NPY_dir)
print(NPY_dir)
normalization_dir = NPY_dir

In [ ]:
import os
import re
import shutil
import numpy as np

# =========================
# ✅ USER SETTINGS
# =========================
normalization_dir = normalization_dir  # 원본 npy들이 있는 폴더 (예: .../output_3D_256_REV/NPY)
output_dir = output_dir                # dataset을 만들 output 폴더

TEST_CASES = {
    "000WL","011WL","014WL","018WL","020WL","023WL","026WL","038WL","053WL","059WL",
    "060WL","065WL","066WL","069WL","078WL","082WL","091WL","093WL","096WL","098WL",
    "000WR","004WR","011WR","020WR","023WR","038WR","039WR","043WR","055WR","056WR",
    "060WR","067WR","069WR","071WR","072WR","074WR","081WR","086WR","090WR","094WR",
}
EXCLUDE_CASES = {"095WL","089WR"}  # Train/Test 모두에서 제외

OVERWRITE = True  # True면 output_dir/dataset 폴더를 통째로 지우고 새로 생성

# =========================
# Helpers
# =========================
CASE_ID_RE = re.compile(r"^(\d{3}W[LR])")  # 예: 098WL, 000WR ...

def extract_case_id(filename: str) -> str | None:
    m = CASE_ID_RE.match(filename)
    return m.group(1) if m else None

# =========================
# Output dirs
# =========================
dataset_output = os.path.join(output_dir, "dataset")

if OVERWRITE and os.path.exists(dataset_output):
    shutil.rmtree(dataset_output)

Train_CT_path  = os.path.join(dataset_output, "Train", "CT_and_Contour")
Train_Dose_path= os.path.join(dataset_output, "Train", "Dose")
Test_CT_path   = os.path.join(dataset_output, "Test",  "CT_and_Contour")
Test_Dose_path = os.path.join(dataset_output, "Test",  "Dose")

os.makedirs(Train_CT_path, exist_ok=True)
os.makedirs(Train_Dose_path, exist_ok=True)
os.makedirs(Test_CT_path, exist_ok=True)
os.makedirs(Test_Dose_path, exist_ok=True)

# =========================
# Main loop
# =========================
folders = sorted([
    d for d in os.listdir(normalization_dir)
    if os.path.isdir(os.path.join(normalization_dir, d))
])

global_idx = 0

saved_train = 0
saved_test = 0
skipped_exclude = 0
skipped_badname = 0
skipped_badshape = 0
skipped_loaderr = 0

found_test_cases = set()

for folder in folders:
    folder_path = os.path.join(normalization_dir, folder)
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".npy")])

    for f in files:
        case_id = extract_case_id(f)

        if case_id is None:
            skipped_badname += 1
            print(f"[SKIP:badname] {folder}/{f}")
            continue

        if case_id in EXCLUDE_CASES:
            skipped_exclude += 1
            continue

        src_path = os.path.join(folder_path, f)
        try:
            data = np.load(src_path)
        except Exception as e:
            skipped_loaderr += 1
            print(f"[SKIP:loaderr] {src_path} -> {e}")
            continue

        # 기대: (H, W, D, 10)
        if data.ndim != 4 or data.shape[-1] != 10:
            skipped_badshape += 1
            print(f"[SKIP:badshape] {src_path} shape={data.shape}")
            continue

        ct_data = data[..., 0:9]          # (H,W,D,9)
        dose_data = data[..., 9]          # (H,W,D)  <= squeeze와 동일 효과

        is_test = (case_id in TEST_CASES)
        if is_test:
            ct_dir, dose_dir = Test_CT_path, Test_Dose_path
            saved_test += 1
            found_test_cases.add(case_id)
        else:
            ct_dir, dose_dir = Train_CT_path, Train_Dose_path
            saved_train += 1

        # 기존 코드와 유사한 네이밍(CT/Dose pair가 정렬로 잘 맞도록)
        ct_filename = f"{global_idx:06d}_CT_and_Contour_{f}"
        dose_filename = f"{global_idx:06d}_Dose_{f}"

        np.save(os.path.join(ct_dir, ct_filename), ct_data)
        np.save(os.path.join(dose_dir, dose_filename), dose_data)

        global_idx += 1

# =========================
# Summary
# =========================
missing_test_cases = sorted(TEST_CASES - found_test_cases)

print("\n================= SUMMARY =================")
print(f"Saved Train: {saved_train}")
print(f"Saved Test : {saved_test}")
print(f"Skipped (exclude): {skipped_exclude}  -> {sorted(EXCLUDE_CASES)}")
print(f"Skipped (badname): {skipped_badname}")
print(f"Skipped (badshape): {skipped_badshape}")
print(f"Skipped (loaderr): {skipped_loaderr}")

if missing_test_cases:
    print("\n[WARN] Requested TEST cases not found in source:")
    print(missing_test_cases)

print("Done ✅")

## make dataset.json for text diffusion model

text 데이터는 llava의 가중치 모델을 이용해서 텍스트 묘사 및 치료 조건에 대해서 설명문을 작성하는 것을 목표로 한다.

In [ ]:
print(f"output dataset path: {dataset_output}")